# 🧰 quant-kit — Kaggle Benchmark Suite

**Free benchmarks using Kaggle's T4 GPU.**  
Runs: Speed (PP/TG), Perplexity, TruthfulQA, GPQA Diamond, ARC Challenge, HellaSwag, GSM8K, Winogrande  
Results auto-upload to your HuggingFace repo.

### Setup
1. Add your HuggingFace token as a Kaggle Secret named `HF_TOKEN`
2. Set `HF_REPO` and `QUANT_TYPE` below
3. Run All Cells

> Runtime: Enable **GPU T4 x2** in Settings → Accelerator

In [ ]:
# ── CONFIG — Edit these ────────────────────────────────────────────────
HF_REPO     = "Dhptl/gemma-4-12b-it-GGUF"   # Your HuggingFace model repo
QUANT_TYPE  = "Q4_K_M"                        # Which quant to benchmark
RUN_SPEED   = True    # llama-bench speed test
RUN_PPL     = True    # Perplexity on WikiText-2
RUN_EVAL    = True    # lm-evaluation-harness tasks
EVAL_TASKS  = "truthfulqa_mc2,gpqa_diamond,arc_challenge,hellaswag,gsm8k,winogrande"
# ──────────────────────────────────────────────────────────────────────

In [ ]:
# ── Install dependencies ───────────────────────────────────────────────
import subprocess, os, sys

print("Installing dependencies...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "huggingface_hub", "lm-eval[api]", "psutil"], check=True)

# Install llama-cpp-python with CUDA support
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "llama-cpp-python",
    "--extra-index-url", "https://abetlen.github.io/llama-cpp-python/whl/cu121"],
    check=True)

# Download llama.cpp pre-built binaries for speed/perplexity benchmarks
import urllib.request
LLAMA_VERSION = "b5262"
LLAMA_URL = f"https://github.com/ggerganov/llama.cpp/releases/download/{LLAMA_VERSION}/llama-{LLAMA_VERSION}-bin-ubuntu-x64.zip"
print(f"Downloading llama.cpp {LLAMA_VERSION}...")
urllib.request.urlretrieve(LLAMA_URL, "/tmp/llama.zip")
subprocess.run(["unzip", "-q", "-o", "/tmp/llama.zip", "-d", "/tmp/llama"], check=True)
subprocess.run(["chmod", "+x"] + [str(p) for p in __import__('pathlib').Path("/tmp/llama").glob("**/llama-*")], check=False)

# Find the binaries
from pathlib import Path
llama_dir = next(Path("/tmp/llama").rglob("llama-bench")).parent
LLAMA_BENCH      = str(llama_dir / "llama-bench")
LLAMA_PERPLEXITY = str(llama_dir / "llama-perplexity")
print(f"llama.cpp binaries at: {llama_dir}")

In [ ]:
# ── Download GGUF from HuggingFace ─────────────────────────────────────
from kaggle_secrets import UserSecretsClient
from huggingface_hub import hf_hub_download, HfApi
import os

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN

model_name = HF_REPO.split("/")[1]
base_name  = model_name.replace("-GGUF", "")
gguf_file  = f"{base_name}-{QUANT_TYPE}.gguf"
model_path = f"/kaggle/working/{gguf_file}"

print(f"Downloading {gguf_file} from {HF_REPO}...")
hf_hub_download(
    repo_id=HF_REPO,
    filename=gguf_file,
    local_dir="/kaggle/working",
    token=HF_TOKEN
)
size_gb = Path(model_path).stat().st_size / 1e9
print(f"Downloaded: {gguf_file} ({size_gb:.2f} GB)")

In [ ]:
# ── 1. Speed Benchmark (llama-bench) ──────────────────────────────────
import json, subprocess, time

speed_results = []

if RUN_SPEED:
    print("\n" + "="*50)
    print("  Speed Benchmark (llama-bench)")
    print("="*50)

    for ctx in [128, 512, 2048]:
        print(f"  Testing context={ctx}...")
        cmd = [
            LLAMA_BENCH, "-m", model_path,
            "-ngl", "99", "-p", str(ctx), "-n", "128", "-r", "3",
            "--output", "json"
        ]
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=600)
        pp_tps, tg_tps = None, None
        if r.returncode == 0 and r.stdout.strip():
            try:
                data = json.loads(r.stdout)
                for entry in data:
                    if isinstance(entry, dict):
                        if entry.get("n_prompt", 0) > 0 and entry.get("n_gen", 0) == 0:
                            pp_tps = round(float(entry.get("avg_ts", 0)), 2)
                        elif entry.get("n_gen", 0) > 0 and entry.get("n_prompt", 0) == 0:
                            tg_tps = round(float(entry.get("avg_ts", 0)), 2)
            except Exception: pass
        speed_results.append({"context": ctx, "pp_tok_s": pp_tps, "tg_tok_s": tg_tps})
        print(f"    ctx={ctx}: TG={tg_tps} tok/s  PP={pp_tps} tok/s")

print("Speed benchmark done!")

In [ ]:
# ── 2. Perplexity on WikiText-2 ────────────────────────────────────────
import subprocess, re

ppl_result = None

if RUN_PPL:
    print("\n" + "="*50)
    print("  Perplexity (WikiText-2)")
    print("="*50)

    # Download WikiText-2 test set
    from huggingface_hub import hf_hub_download
    import datasets
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "datasets"])
    ds = datasets.load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
    wiki_text = "\n".join(ds["text"])
    with open("/kaggle/working/wiki.test.raw", "w") as f:
        f.write(wiki_text)
    print("WikiText-2 downloaded")

    cmd = [
        LLAMA_PERPLEXITY,
        "-m", model_path,
        "-f", "/kaggle/working/wiki.test.raw",
        "-c", "512", "-ngl", "99"
    ]
    print("Running perplexity... (this takes ~30-60 min)")
    r = subprocess.run(cmd, capture_output=True, text=True, timeout=7200)

    for line in reversed(r.stderr.splitlines()):
        if "Final estimate" in line and "PPL" in line:
            m = re.search(r"PPL\s*=\s*([\d.]+)", line)
            if m:
                ppl_result = float(m.group(1))
                print(f"  Perplexity = {ppl_result}")
                break
    if not ppl_result:
        print("  Could not parse PPL from output")

In [ ]:
# ── 3. lm-evaluation-harness benchmarks ───────────────────────────────
import subprocess, sys, json
from pathlib import Path

eval_results = {}

if RUN_EVAL:
    print("\n" + "="*50)
    print(f"  lm-eval: {EVAL_TASKS}")
    print("="*50)

    results_dir = Path("/kaggle/working/eval_results")
    results_dir.mkdir(exist_ok=True)

    cmd = [
        sys.executable, "-m", "lm_eval",
        "--model", "gguf",
        "--model_args", f"pretrained={model_path}",
        "--tasks", EVAL_TASKS,
        "--output_path", str(results_dir),
        "--batch_size", "auto",
        "--device", "cuda",
        "--log_samples",
    ]

    print("Running lm-eval (this may take 2-4 hours)...")
    proc = subprocess.run(cmd, capture_output=False, text=True, timeout=18000)

    # Parse results
    for result_file in results_dir.glob("**/*.json"):
        if "results" in result_file.name:
            with open(result_file) as f:
                data = json.load(f)
                for task, metrics in data.get("results", {}).items():
                    key_metric = (
                        metrics.get("acc_norm,none") or
                        metrics.get("acc,none") or
                        metrics.get("exact_match,none")
                    )
                    if key_metric is not None:
                        eval_results[task] = round(key_metric * 100, 2)
                        print(f"  {task}: {eval_results[task]}%")
            break

print("lm-eval done!")

In [ ]:
# ── 4. Collect all results & upload to HuggingFace ─────────────────────
import json, platform
from huggingface_hub import HfApi

output = {
    "model":      HF_REPO,
    "quant":      QUANT_TYPE,
    "platform":   "Kaggle T4 GPU",
    "speed":      speed_results,
    "perplexity": ppl_result,
    "benchmarks": eval_results,
}

result_file = f"/kaggle/working/kaggle_results_{QUANT_TYPE}.json"
with open(result_file, "w") as f:
    json.dump(output, f, indent=2)
print(f"Results saved locally: {result_file}")

# Upload back to HuggingFace repo
api = HfApi(token=HF_TOKEN)
api.upload_file(
    path_or_fileobj=result_file,
    path_in_repo=f"kaggle_results_{QUANT_TYPE}.json",
    repo_id=HF_REPO,
    repo_type="model",
    commit_message=f"Add Kaggle benchmark results for {QUANT_TYPE}"
)
print(f"✅ Results uploaded to https://huggingface.co/{HF_REPO}")
print()
print("=" * 50)
print("  FINAL RESULTS SUMMARY")
print("=" * 50)
print(json.dumps(output, indent=2))